In [25]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [26]:
def softmax(z): # output is add to 1
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def categorical_cross_entropy(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)  # avoid log(0) so that's why we use the np.clip
    return -np.sum(y_true * np.log(y_pred)) / y_true.shape[0]
# here le-15 is scientific notation. e means times to the power of.
# we need this because log(0) is undefined and would break the loss

In [32]:
# Load data
iris = load_iris()
X = iris.data      # (150, 4): 150 flowers, 4 measurements each
y = iris.target    # (150,): 0 = setosa, 1 = versicolor, 2 = virginica

# One-hot encode all labels
num_classes = 3
y_one_hot = np.eye(num_classes)[y]   # (150, 3)

# spliting the dataset in training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y_one_hot, test_size=0.2, random_state=42, stratify=y)  # stratifying keeps the class balance even

# showing the some training data how it look
df = pd.DataFrame(X_train)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # we use fit.transform() method in training because it calculates required values (fit()) and applies
# transformations to this training data
X_test = scaler.transform(X_test)  # why we use transform() method to test data is because the values we use for training it applies here and it use those mean and sd in here .

# weight intialization and training loop
num_features = X_train.shape[1]  # how many features (columns)
n_samples = X_train.shape[0]     # how many data examples (rows)

# intialize the weights (4,3) and bias (1,3)
np.random.seed(0)
W = np.random.randn(4, 3) * 0.01
b = np.zeros((1, 3))

# hyperparameters
lr = 0.1
epochs = 3000

for epoch in range(epochs):
    # forward pass
    z = np.dot(X_train, W) + b
    A = softmax(z)

    # loss - how far the predictions are from the one-hot targets
    loss = categorical_cross_entropy(y_train, A)

    # dz is the error at the logits for every flower, so its output is a matrix of shape (120, 3).
    # Each row is one flower, and each column is one class.
    # Each entry is:
    # dz[i, k] = (A[i, k] - y_train[i, k]) / 120
    # For example, using the earlier flower with A = [0.703, 0.192, 0.105] and a setosa target y = [1, 0, 0],
    # its row in dz would be:
    # [-0.297, 0.192, 0.105] / 120  =  [-0.00248, 0.00160, 0.00088]

    # how to calculate the gradients
    dz = (A - y_train) / n_samples
    dW = np.dot(X_train.T, dz)
    db = np.sum(dz, axis=0, keepdims=True)  # adds the errors down each column, so you get one total per class across all 120 flowers.
    # Why sum: each bias adds 1 to its class's score for every flower, so its effect on the loss is the total of all the flowers' errors for that class.
    # The output has shape (1, 3), with one gradient per bias. Each entry is:
    # db[k] = sum over flowers of dz[i, k]

    # update the parameters
    W -= lr * dW
    b -= lr * db

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss:.4f}")

print("dW:\n", dW)    # slope of the loss for each of the 12 weights
print("db:", db)      # slope of the loss for each of the 3 biases

Epoch 100/3000 - Loss: 0.3212
Epoch 200/3000 - Loss: 0.2522
Epoch 300/3000 - Loss: 0.2123
Epoch 400/3000 - Loss: 0.1854
Epoch 500/3000 - Loss: 0.1660
Epoch 600/3000 - Loss: 0.1514
Epoch 700/3000 - Loss: 0.1401
Epoch 800/3000 - Loss: 0.1310
Epoch 900/3000 - Loss: 0.1235
Epoch 1000/3000 - Loss: 0.1172
Epoch 1100/3000 - Loss: 0.1119
Epoch 1200/3000 - Loss: 0.1074
Epoch 1300/3000 - Loss: 0.1034
Epoch 1400/3000 - Loss: 0.1000
Epoch 1500/3000 - Loss: 0.0969
Epoch 1600/3000 - Loss: 0.0941
Epoch 1700/3000 - Loss: 0.0917
Epoch 1800/3000 - Loss: 0.0895
Epoch 1900/3000 - Loss: 0.0874
Epoch 2000/3000 - Loss: 0.0856
Epoch 2100/3000 - Loss: 0.0839
Epoch 2200/3000 - Loss: 0.0823
Epoch 2300/3000 - Loss: 0.0809
Epoch 2400/3000 - Loss: 0.0795
Epoch 2500/3000 - Loss: 0.0783
Epoch 2600/3000 - Loss: 0.0771
Epoch 2700/3000 - Loss: 0.0761
Epoch 2800/3000 - Loss: 0.0750
Epoch 2900/3000 - Loss: 0.0741
Epoch 3000/3000 - Loss: 0.0732
dW:
 [[ 0.00155532 -0.00105634 -0.00049899]
 [-0.0012376  -0.00018775  0.001425

In [33]:
# forward pass on test data (no training, no gradients)
z_test = np.dot(X_test, W) + b
A_test = softmax(z_test)

# predicted class = highest probability; true class = highest value in one-hot target
pred = np.argmax(A_test, axis=1)
true = np.argmax(y_test, axis=1)

accuracy = np.mean(pred == true) * 100
print(f"Test accuracy: {accuracy:.2f}%")

# inspect one flower
i = 0
print("True species:     ", iris.target_names[true[i]])
print("Predicted probs:  ", np.round(A_test[i], 4))
print("Predicted species:", iris.target_names[pred[i]])

train_pred = np.argmax(softmax(np.dot(X_train, W) + b), axis=1)
train_true = np.argmax(y_train, axis=1)
print(f"Train accuracy: {np.mean(train_pred == train_true) * 100:.2f}%")

Test accuracy: 96.67%
True species:      setosa
Predicted probs:   [0.998 0.002 0.   ]
Predicted species: setosa
Train accuracy: 97.50%
